# Observability & Monitoring — Hands-On

**LLM Engineering · Domain 8 · Roadmap Week 22**

Offline notebook: trace spans, token/cost accounting, p50/p95 metrics, PII redaction, drift and feedback simulation.

## 0. Setup

In [ ]:
%pip install -q numpy
import re, hashlib
from dataclasses import dataclass, field
import numpy as np
rng = np.random.RandomState(45)
print("ok")

## 1. Token and cost accountant

In [ ]:
PRICES = {"small": (0.15, 0.60), "strong": (2.50, 10.00)}
def estimate_tokens(text): return max(1, len(text.split()) * 4 // 3)
def cost_usd(model, prompt, completion):
    it, ot = estimate_tokens(prompt), estimate_tokens(completion)
    pin, pout = PRICES[model]
    return {"input_tokens": it, "output_tokens": ot, "cost_usd": (it*pin + ot*pout)/1_000_000}
print(cost_usd("small", "summarize this support ticket", "short answer"))

## 2. Minimal trace tree

In [ ]:
@dataclass
class Span:
    name: str
    attrs: dict = field(default_factory=dict)
    children: list = field(default_factory=list)
    def add(self, name, **attrs):
        child = Span(name, attrs); self.children.append(child); return child

def show(span, indent=0):
    print("  "*indent + span.name, span.attrs)
    for c in span.children: show(c, indent+1)
root = Span("request", {"trace_id": "obs-1", "tenant": "demo"})
root.add("retrieval", query="refund policy", doc_ids=["d7","d9"], latency_ms=42)
root.add("llm.call", model="small", prompt_version="qa-v4", **cost_usd("small", "refund policy", "answer"), latency_ms=180)
show(root)

## 3. p50 and p95 latency over simulated traffic

In [ ]:
latency = rng.lognormal(mean=5.25, sigma=0.35, size=200)
tokens = rng.randint(250, 2200, size=200)
errors = rng.rand(200) < 0.035
print("p50_ms", round(float(np.percentile(latency, 50)), 1))
print("p95_ms", round(float(np.percentile(latency, 95)), 1))
print("error_rate", round(float(errors.mean()), 3))
print("avg_tokens", round(float(tokens.mean()), 1))

## 4. PII-safe prompt logging

In [ ]:
EMAIL = re.compile(r"[\w.%-]+@[\w.-]+\.[A-Za-z]{2,}")
PHONE = re.compile(r"\d{3}[-.]?\d{3}[-.]?\d{4}")
def redact(text): return PHONE.sub("<PHONE>", EMAIL.sub("<EMAIL>", text))
def prompt_hash(template): return hashlib.sha256(template.encode()).hexdigest()[:10]
raw = "User alice@example.com says call 555-123-9999 about claim 42"
print(redact(raw)); print("template_hash", prompt_hash("Answer from context: {context}"))

## 5. Drift detection on embedding-like vectors

In [ ]:
baseline = rng.normal(0, 1, size=(100, 8))
current = rng.normal(0.35, 1, size=(100, 8))
shift = np.linalg.norm(current.mean(axis=0) - baseline.mean(axis=0))
print("centroid_shift", round(float(shift), 3), "alert", bool(shift > 0.60))

## 6. Feedback joins back to traces

In [ ]:
traces = [{"trace_id": f"t{i}", "prompt_version": "qa-v4" if i<5 else "qa-v5"} for i in range(10)]
feedback = {"t1": -1, "t3": 1, "t6": -1, "t7": -1, "t8": 1}
by_version = {}
for t in traces:
    if t["trace_id"] in feedback:
        by_version.setdefault(t["prompt_version"], []).append(feedback[t["trace_id"]])
print({k: round(sum(v)/len(v), 2) for k, v in by_version.items()})

## 7. Exercises
1. Add a `tool.call` span with error status and retry count.
2. Compute cost per tenant and alert when a budget is exceeded.
3. Add a redactor for 16-digit card-like numbers.
4. Slice p95 latency by model.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Observability and Monitoring`
- Snippets: `04 Code Snippets/LLM/LLM Trace and Cost Ledger`, `.../PII Safe LLM Metrics Dashboard`